# 05 -- Farmer movement (Level 2, first increment)

Level 1's `wheat_loop_agent` only ever tends its single starting tile
(farmer never moves) -- see `PROGRESS.md`'s 2026-08-13 entry, where the
real leaderboard replay against `HechtJP` showed our farmer doing `PASS`
on 91.7% of turns while the winner moves and tends multiple tiles every
day.

This notebook builds the first increment: a farmer that cycles through a
closed loop of tiles instead of tending one. Route (see `PROGRESS.md`
roadmap): confirm movement mechanics -> pick a traversal strategy ->
prototype here -> validate against the single-tile baseline via
`evaluate.py` -> graduate to `src/`.

## Why a day supports more than 1 tile, and why 2x2 first

There are `turnsPerDay = 24` turns per in-game day, and each turn is
exactly one action -- moving and tending a tile are mutually exclusive.
Visiting a new tile costs roughly 1 move + 1 tend action, so a single
farmer can service at most about `24 / 2 ~= 12` tiles per day -- well
above 1, but also a hard ceiling worth respecting when picking a loop
size.

Starting simple: a 2x2 loop (4 tiles) around the shed spawn point
`(4,4)` -- small enough to reason about by hand, big enough to prove
"move and tend multiple tiles" actually beats the single-tile baseline
before growing the loop further.

In [1]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.evaluate import evaluate, run_episodes
from kaggriculture_bot.strategies.baseline import wheat_loop_agent


## Confirming the movement axis empirically

`README.md` documents `NORTH`/`SOUTH`/`EAST`/`WEST` as one-cell moves but
never states which axis each corresponds to -- don't assume it, check it
(this project's rule of verifying data semantics instead of guessing).
The farmer spawns at `(4,4)`; move it `NORTH` once and compare its
position before/after.

In [2]:
def probe_agent(obs):
    return {"farmer": ["NORTH"], "hands": [], "market": []}


env_probe = make("kaggriculture", configuration={"episodeSteps": 5}, debug=True)
env_probe.run([probe_agent, "random"])

before = env_probe.steps[0][0]["observation"]["farms"][0]["farmer"]
after = env_probe.steps[1][0]["observation"]["farms"][0]["farmer"]
print("antes:", before, "despues:", after)


antes: [4, 4] despues: [4, 3]


Result: `antes: [4, 4] despues: [4, 3]` -- `NORTH` decrements `y`. So:

- NORTH -> dy = -1
- SOUTH -> dy = +1
- EAST -> dx = +1
- WEST -> dx = -1

## The 2x2 loop agent

Board layout (`README.md`): 10x10 split into four 5x5 quadrants; the
farmer spawns at `(4,4)`, the NW quadrant's corner closest to the shed.
`LOOP_ORDER_2X2` is a fixed cycle through the 4 tiles closest to the
spawn point: `(4,4) -> (4,3) -> (3,3) -> (3,4) -> (4,4)`. Since the whole
board state (including the farmer's current position) is already in
`obs` every turn, the agent needs no memory between calls -- the next
target tile is always just `LOOP_ORDER_2X2[current_position]`.

Priority per turn, extending `wheat_loop_agent`'s own logic: harvest >
water > plant > move to the next tile in the loop. The "move" branch also
covers "this tile already has everything it needs today", not just "tile
is empty and I'm out of seed" -- see the bug note below for why that
distinction mattered.

In [3]:
LOOP_ORDER_2X2 = {
    (4, 4): (4, 3),
    (4, 3): (3, 3),
    (3, 3): (3, 4),
    (3, 4): (4, 4),
}
DIRECTIONS = {(0, -1): "NORTH", (0, 1): "SOUTH", (1, 0): "EAST", (-1, 0): "WEST"}


def wheat_loop_2x2_agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    if obs["step"] == 0 or wheat_seeds <= 0:
        market.append(["BUY_SEED", "WHEAT", 1])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_2X2[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": [], "market": market}


## Bug found: the farmer got stuck on the first tile

First working draft evaluated as a **loss**: `wheat_loop_agent` (1 tile)
beat this movement version 20/20. Inspecting a replay (not guessing)
showed why: once a tile held a growing crop, "already watered today,
nothing more to do" fell through to a plain `PASS` instead of to the
movement branch -- so the farmer camped the first tile it moved to
(`(4,3)`) for its entire ~20-turn/day idle stretch and never reached the
other 3 tiles. Fix: that case now returns the same move-to-next-tile
action as the empty-tile/out-of-seed case (the two `else: farmer =
[direction]` branches in the cell above).

Full before/after numbers are logged in `PROGRESS.md` (2026-08-19).

In [4]:
import json

env_selfplay = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
env_selfplay.run([wheat_loop_2x2_agent, wheat_loop_2x2_agent])

with open("../replays/05_farmer_movement_2x2_selfplay.json", "w", encoding="utf-8") as f:
    json.dump(env_selfplay.toJSON(), f)

print("steps recorded:", len(env_selfplay.steps))


steps recorded: 720


## Validating against the single-tile baseline

Per the roadmap, the comparison that matters is against `wheat_loop_agent`,
not just the built-in `"random"`/`"starter"` agents.

In [5]:
print(evaluate(wheat_loop_agent, wheat_loop_2x2_agent, n=20))


{'win_rate': 0.0, 'avg_money': 3326.0}


### Sanity check: is `n` inside one `evaluate()` call actually independent?

The `n=20` run above came back bit-for-bit identical across all 20
episodes, which shouldn't happen if there's real per-episode randomness
(weed spawns, town shop unlocks). Checked by calling `evaluate()` twice
more, separately -- each call creates its own fresh `env` via `make()`:

In [6]:
print(evaluate(wheat_loop_agent, wheat_loop_2x2_agent, n=10))
print(evaluate(wheat_loop_agent, wheat_loop_2x2_agent, n=10))


{'win_rate': 0.0, 'avg_money': 3371.0}
{'win_rate': 0.0, 'avg_money': 3367.0}


### Conclusion

All three calls agree in direction: `wheat_loop_agent` (1 tile) loses
20/20 to `wheat_loop_2x2_agent`, with the baseline's `avg_money` landing
around 3300-3385 depending on the call vs. ~4293 for the movement version
(roughly +29%). The lack of variance *within* a single call's `n` looks
like an `evaluate.py`/harness quirk (the reused `env` object doesn't
reseed between its `.run()` calls) rather than a property of these two
agents specifically -- noted in `PROGRESS.md` as something to check
before trusting a single large-`n` call in future evaluations.

Graduated to
`src/kaggriculture_bot/strategies/baseline.py::wheat_loop_2x2_agent`, with
unit tests in `tests/test_baseline.py` and a `RESOURCES.md` entry. Not yet
wired into `agent/main.py` (`scripts/build_agent.py`'s `STRATEGY` constant
still points at `wheat_loop_agent`) -- see `PROGRESS.md`'s Next steps.

In [7]:
LOOP_ORDER_12 = {
    (4, 4): (3, 4),
    (3, 4): (2, 4),
    (2, 4): (1, 4),
    (1, 4): (0, 4),
    (0, 4): (0, 3),
    (0, 3): (1, 3),
    (1, 3): (2, 3),
    (2, 3): (2, 2),
    (2, 2): (3, 2),
    (3, 2): (3, 3),
    (3, 3): (4 ,3),
    (4, 3): (4, 4),
}
DIRECTIONS = {(0, -1): "NORTH", (0, 1): "SOUTH", (1, 0): "EAST", (-1, 0): "WEST"}


def wheat_loop_12_agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    if obs["step"] == 0 or wheat_seeds <= 0:
        market.append(["BUY_SEED", "WHEAT", 1])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": [], "market": market}


In [8]:
evaluate(wheat_loop_2x2_agent, wheat_loop_12_agent, n=20)

{'win_rate': 0.0, 'avg_money': 4301.0}

In [9]:
evaluate(wheat_loop_2x2_agent, wheat_loop_12_agent, n=20)

{'win_rate': 0.0, 'avg_money': 4413.0}